# 面试问题：Agent 工具供应链怎样验证签名 Manifest、制品摘要和完整 Provenance？

        ## 可直接复述的回答主线

        1. 工具加载不能只检查 manifest 里有 signature 字段，必须用受信公钥重新计算并验证签名覆盖的规范载荷。
2. 签名载荷至少绑定工具名、版本、publisher、key_id、artifact digest、schema hash 和声明权限。
3. 加载顺序应验证制品 SHA-256、公钥与 publisher 绑定、撤销状态、数学签名、schema allowlist 和权限上限。
4. 任何一步失败都要 fail closed，并在 provenance ledger 记录 package、摘要、key、检查链和原因。
5. 修改权限但保留旧签名仍然有 signature 字段，真实验签会因规范载荷变化而拒绝。
6. 生产应使用 Ed25519/Sigstore、TUF 元数据、透明日志、KMS 密钥、时间戳、阈值签名和撤销传播。

        后续实验会在同一批输入上依次展示朴素基线、手写核心机制、中间过程、失败修正和生产边界。

## 1. 真实案例与输入预览

案例包含七个工具包：天气与文档为合法包，支付权限越界，搜索制品被替换，CRM 使用已撤销密钥，日历签名被改写，分析工具使用未批准 schema。Notebook 用极小教学 RSA 参数真实执行模幂验签；密钥尺寸完全不安全，只用于解释验证链。

In [1]:
import hashlib  # 对制品、schema 和规范 manifest 计算真实 SHA-256。
import json  # 生成字段顺序稳定的签名载荷。
toy_publishers = {"acme-tools": {"key_id": "acme-1", "p": 61, "q": 53, "e": 17}, "lab-tools": {"key_id": "lab-1", "p": 47, "q": 71, "e": 79}, "legacy-tools": {"key_id": "legacy-1", "p": 43, "q": 59, "e": 13}}  # 定义仅供教学的极小 RSA 发布者私钥材料。
for publisher, key in toy_publishers.items():  # 为三个教学发布者计算 RSA 公私参数。
    key["n"] = key["p"] * key["q"]  # 计算 RSA 模数。
    phi = (key["p"] - 1) * (key["q"] - 1)  # 计算欧拉函数供私钥求逆。
    key["d"] = pow(key["e"], -1, phi)  # 计算离线发布者签名指数。
public_keyring = {key["key_id"]: {"publisher": publisher, "n": key["n"], "e": key["e"]} for publisher, key in toy_publishers.items()}  # 构造运行时只含公钥的受信 keyring。
revoked_key_ids = {"legacy-1"}  # 模拟撤销服务已撤销旧 CRM 发布密钥。
schema_texts = {"weather": "city:string->forecast:string", "payments": "invoice_id:string->status:string", "search": "query:string->hits:list", "crm": "customer_id:string->profile:object", "calendar": "date:string->events:list", "docs": "path:string->content:string", "analytics": "metric:string->series:list"}  # 定义七个工具的批准接口 schema。
approved_schema_hashes = {name: hashlib.sha256(schema.encode("utf-8")).hexdigest() for name, schema in schema_texts.items()}  # 为每个批准 schema 计算 allowlist 摘要。
approved_permissions = {"weather": {"network.weather.read"}, "payments": {"payments.read"}, "search": {"search.read"}, "crm": {"crm.read"}, "calendar": {"calendar.read"}, "docs": {"docs.read"}, "analytics": {"metrics.read"}}  # 定义工具运行时最大权限上限。
def canonical_manifest_bytes(manifest):  # 生成签名覆盖的规范 manifest 字节。
    payload = {name: value for name, value in manifest.items() if name != "signature"}  # 排除签名字段自身并保留所有安全属性。
    return json.dumps(payload, sort_keys=True, separators=(",", ":"), ensure_ascii=False).encode("utf-8")  # 固定字段顺序、空白和 UTF-8 编码。
def rsa_message_integer(message, modulus):  # 把 SHA-256 消息摘要映射到教学 RSA 模数域。
    digest = hashlib.sha256(message).digest()  # 计算规范载荷的完整 SHA-256。
    return int.from_bytes(digest, "big") % modulus  # 取模得到教学 RSA 可签名整数。
def sign_manifest(manifest, publisher):  # 使用离线发布者私钥真实执行 RSA 模幂签名。
    key = toy_publishers[publisher]  # 读取教学发布者私钥参数。
    message_integer = rsa_message_integer(canonical_manifest_bytes(manifest), key["n"])  # 计算待签名规范载荷整数。
    return pow(message_integer, key["d"], key["n"])  # 用私钥指数生成真实可验证签名整数。
def build_package(package_id, name, version, publisher, permissions, artifact, expected, schema_hash=None):  # 创建带真实摘要和签名的工具包夹具。
    key = toy_publishers[publisher]  # 读取发布者 key_id 和教学私钥。
    manifest = {"name": name, "version": version, "publisher": publisher, "key_id": key["key_id"], "artifact_sha256": hashlib.sha256(artifact).hexdigest(), "schema_sha256": schema_hash or approved_schema_hashes[name], "permissions": sorted(permissions)}  # 构造签名覆盖的全部供应链声明。
    manifest["signature"] = sign_manifest(manifest, publisher)  # 对规范 manifest 执行真实 RSA 签名。
    return {"id": package_id, "manifest": manifest, "delivered_artifact": artifact, "expected": expected}  # 返回待加载包和期望决策。
packages = [build_package("pkg-01", "weather", "1.4.2", "acme-tools", {"network.weather.read"}, b"weather-client-v1.4.2:GET-/forecast", True), build_package("pkg-02", "payments", "2.1.0", "acme-tools", {"payments.write"}, b"payments-client-v2.1.0:POST-/capture", False), build_package("pkg-03", "search", "3.0.1", "lab-tools", {"search.read"}, b"search-client-v3.0.1:query", False), build_package("pkg-04", "crm", "1.8.0", "legacy-tools", {"crm.read"}, b"crm-client-v1.8.0:get-profile", False), build_package("pkg-05", "calendar", "2.0.0", "lab-tools", {"calendar.read"}, b"calendar-client-v2.0.0:list-events", False), build_package("pkg-06", "docs", "4.2.0", "lab-tools", {"docs.read"}, b"docs-client-v4.2.0:read", True), build_package("pkg-07", "analytics", "1.1.0", "acme-tools", {"metrics.read"}, b"analytics-client-v1.1.0:series", False, schema_hash=hashlib.sha256(b"metric:string->shell:string").hexdigest())]  # 定义七个包含有效和多种失效语义的工具包。
packages[2]["delivered_artifact"] = b"search-client-v3.0.1:query;exfiltrate-env"  # 在签名后替换搜索制品以制造摘要不匹配。
calendar_key = public_keyring[packages[4]["manifest"]["key_id"]]  # 读取日历签名对应公钥。
packages[4]["manifest"]["signature"] = (packages[4]["manifest"]["signature"] + 1) % calendar_key["n"]  # 改写日历签名整数但保留 signature 字段。
print("教学实验输入：七个工具供应链包")  # 标记下方为离线安全验证案例。
print("包         name/version       publisher       key          permissions                 sig存在 expected")  # 输出包清单表头。
for package in packages:  # 逐包展示可读 provenance 字段。
    manifest = package["manifest"]  # 读取当前签名 manifest。
    print(f"{package['id']:<10} {manifest['name']}/{manifest['version']:<10} {manifest['publisher']:<15} {manifest['key_id']:<12} {str(manifest['permissions']):<27} {str(bool(manifest['signature'])):>7} {str(package['expected']):>8}")  # 输出当前包的发布者、权限和期望。
print("注意：toy RSA模数=", {key_id: value["n"] for key_id, value in public_keyring.items()}, "仅供教学，不具备安全强度。")  # 明确极小 RSA 不能用于生产。

教学实验输入：七个工具供应链包
包         name/version       publisher       key          permissions                 sig存在 expected
pkg-01     weather/1.4.2      acme-tools      acme-1       ['network.weather.read']       True     True
pkg-02     payments/2.1.0      acme-tools      acme-1       ['payments.write']             True    False
pkg-03     search/3.0.1      lab-tools       lab-1        ['search.read']                True    False
pkg-04     crm/1.8.0      legacy-tools    legacy-1     ['crm.read']                   True    False
pkg-05     calendar/2.0.0      lab-tools       lab-1        ['calendar.read']              True    False
pkg-06     docs/4.2.0      lab-tools       lab-1        ['docs.read']                  True     True
pkg-07     analytics/1.1.0      acme-tools      acme-1       ['metrics.read']               True    False
注意：toy RSA模数= {'acme-1': 3233, 'lab-1': 3337, 'legacy-1': 2537} 仅供教学，不具备安全强度。


## 2. Baseline / 基线：只检查 `signature` 字段非空

七个包都带非零整数，因此基线全部加载。它无法发现制品替换、撤销密钥、签名篡改、权限越界和 schema 漂移。

In [2]:
baseline_rows = []  # 保存七个包的浅层签名存在性决策。
for package in packages:  # 对同一批包只检查 signature 字段。
    signature_present = bool(package["manifest"].get("signature"))  # 错误地把非空字段当作供应链可信。
    loaded = signature_present  # 有字段就直接加载工具代码。
    baseline_rows.append({"id": package["id"], "loaded": loaded, "correct": loaded == package["expected"], "reason": "signature_present"})  # 保存加载和期望对照。
baseline_accuracy = sum(row["correct"] for row in baseline_rows) / len(baseline_rows)  # 计算存在性基线决策准确率。
print("Baseline 签名存在性检查")  # 标记下表没有真实密码学和 provenance 验证。
print("包         loaded  expected  correct  reason")  # 输出基线结果表头。
for row, package in zip(baseline_rows, packages):  # 逐包展示危险放行。
    print(f"{row['id']:<10} {str(row['loaded']):>6} {str(package['expected']):>9} {str(row['correct']):>8}  {row['reason']}")  # 输出当前包的浅层决策。

Baseline 签名存在性检查
包         loaded  expected  correct  reason
pkg-01       True      True     True  signature_present
pkg-02       True     False    False  signature_present
pkg-03       True     False    False  signature_present
pkg-04       True     False    False  signature_present
pkg-05       True     False    False  signature_present
pkg-06       True      True     True  signature_present
pkg-07       True     False    False  signature_present


## 3. 底层实现：摘要、公钥绑定、撤销、RSA 验签、Schema 与权限门禁

验证器只持有 `(n,e)` 公钥。`pow(signature,e,n)` 必须等于规范 manifest 的 SHA-256 映射值；随后还要验证 schema allowlist 与声明权限子集。

In [3]:
def verify_package(package):  # 对单个工具包执行完整 fail-closed 供应链验证。
    manifest = package["manifest"]  # 读取待验证声明。
    trace = []  # 保存每一步检查结果和中间摘要。
    actual_artifact_digest = hashlib.sha256(package["delivered_artifact"]).hexdigest()  # 对实际交付制品重新计算 SHA-256。
    artifact_match = actual_artifact_digest == manifest["artifact_sha256"]  # 比较交付内容与签名声明摘要。
    trace.append({"check": "artifact_sha256", "passed": artifact_match, "actual": actual_artifact_digest[:12], "expected": manifest["artifact_sha256"][:12]})  # 记录制品完整性证据。
    if not artifact_match:  # 在执行任何工具代码前拒绝制品替换。
        return False, "artifact_digest_mismatch", trace  # 返回明确失败原因和已完成检查。
    key = public_keyring.get(manifest["key_id"])  # 根据 key_id 读取受信公钥。
    key_known = key is not None  # 检查 key_id 是否存在于受信根。
    trace.append({"check": "trusted_key", "passed": key_known, "key_id": manifest["key_id"]})  # 记录公钥解析结果。
    if not key_known:  # 未知 key 不允许回退为自带公钥。
        return False, "unknown_key", trace  # 以 fail closed 结束加载。
    publisher_bound = key["publisher"] == manifest["publisher"]  # 检查公钥所有者与声明 publisher 一致。
    trace.append({"check": "publisher_binding", "passed": publisher_bound, "key_publisher": key["publisher"], "claimed": manifest["publisher"]})  # 记录身份绑定证据。
    if not publisher_bound:  # 阻止合法公钥替其他 publisher 背书。
        return False, "publisher_mismatch", trace  # 返回发布者不匹配。
    key_active = manifest["key_id"] not in revoked_key_ids  # 查询本地撤销快照。
    trace.append({"check": "key_revocation", "passed": key_active, "key_id": manifest["key_id"]})  # 记录密钥有效状态。
    if not key_active:  # 已撤销签名即使数学正确也不可接受。
        return False, "revoked_key", trace  # 阻止旧 CRM 密钥加载。
    expected_integer = rsa_message_integer(canonical_manifest_bytes(manifest), key["n"])  # 对实际规范 manifest 重新计算消息整数。
    recovered_integer = pow(int(manifest["signature"]), key["e"], key["n"])  # 使用受信公钥执行真实 RSA 模幂验签。
    signature_valid = recovered_integer == expected_integer  # 比较验签恢复值和重新计算摘要。
    trace.append({"check": "rsa_signature", "passed": signature_valid, "recovered": recovered_integer, "expected": expected_integer})  # 保存数学验签中间量。
    if not signature_valid:  # 拒绝签名整数或任何签名载荷篡改。
        return False, "invalid_signature", trace  # 返回真实密码学验证失败。
    approved_schema = approved_schema_hashes.get(manifest["name"])  # 读取当前工具的批准 schema 摘要。
    schema_valid = approved_schema is not None and manifest["schema_sha256"] == approved_schema  # 检查签名 schema 是否在 allowlist。
    trace.append({"check": "schema_allowlist", "passed": schema_valid, "actual": manifest["schema_sha256"][:12], "approved": approved_schema[:12] if approved_schema else None})  # 记录 schema provenance。
    if not schema_valid:  # 阻止签名有效但接口被替换的工具。
        return False, "schema_not_approved", trace  # 返回 schema 门禁失败。
    requested_permissions = set(manifest["permissions"])  # 读取签名覆盖的权限集合。
    allowed_permissions = approved_permissions.get(manifest["name"], set())  # 读取运行时权限上限。
    permissions_valid = requested_permissions <= allowed_permissions  # 要求声明权限是 allowlist 子集。
    trace.append({"check": "permission_subset", "passed": permissions_valid, "requested": sorted(requested_permissions), "allowed": sorted(allowed_permissions)})  # 记录权限差异证据。
    if not permissions_valid:  # 阻止签名发布者自行扩大生产权限。
        return False, "permission_escalation", trace  # 返回最小权限门禁失败。
    return True, "verified", trace  # 全部供应链和策略检查通过后允许加载。
first_verified, first_reason, first_trace = verify_package(packages[0])  # 对合法天气工具展示完整验证链。
print("pkg-01完整 Provenance 验证链")  # 标记下方展示每个检查和真实 RSA 中间值。
for check in first_trace:  # 逐项展示摘要、公钥、签名、schema 和权限。
    print(check)  # 输出当前验证步骤的证据。
print("pkg-01最终决策=", first_verified, first_reason)  # 展示只有全部通过才加载。

pkg-01完整 Provenance 验证链
{'check': 'artifact_sha256', 'passed': True, 'actual': 'af70bd78c9d7', 'expected': 'af70bd78c9d7'}
{'check': 'trusted_key', 'passed': True, 'key_id': 'acme-1'}
{'check': 'publisher_binding', 'passed': True, 'key_publisher': 'acme-tools', 'claimed': 'acme-tools'}
{'check': 'key_revocation', 'passed': True, 'key_id': 'acme-1'}
{'check': 'rsa_signature', 'passed': True, 'recovered': 181, 'expected': 181}
{'check': 'schema_allowlist', 'passed': True, 'actual': '9e5ca38f544e', 'approved': '9e5ca38f544e'}
{'check': 'permission_subset', 'passed': True, 'requested': ['network.weather.read'], 'allowed': ['network.weather.read']}
pkg-01最终决策= True verified


## 4. 逐工具包结果与结果解读

每个拒绝包只显示已经执行到的检查链，便于定位是制品、密钥、签名、schema 还是权限问题；合法包必须经过全部六类检查。

In [4]:
corrected_rows = []  # 保存七个包的完整验证结果。
provenance_ledger = []  # 保存不含私钥和制品正文的审计账本。
for package in packages:  # 对同一批工具包执行完整验证。
    loaded, reason, trace = verify_package(package)  # 重新计算摘要、签名与策略检查。
    manifest = package["manifest"]  # 读取审计需要的已声明元数据。
    corrected_rows.append({"id": package["id"], "name": manifest["name"], "loaded": loaded, "reason": reason, "checks": len(trace), "correct": loaded == package["expected"]})  # 保存逐包决策和检查深度。
    provenance_ledger.append({"package_id": package["id"], "name": manifest["name"], "version": manifest["version"], "publisher": manifest["publisher"], "key_id": manifest["key_id"], "artifact_digest": manifest["artifact_sha256"][:16], "decision": reason, "check_chain": [item["check"] for item in trace]})  # 保存可追溯但不泄露私钥的账本。
corrected_accuracy = sum(row["correct"] for row in corrected_rows) / len(corrected_rows)  # 计算完整验证决策准确率。
print("包         name        baseline  verified  reason                       checks  correct")  # 输出同数据逐包对照表头。
for baseline, corrected in zip(baseline_rows, corrected_rows):  # 逐包比较存在性与完整验证。
    print(f"{corrected['id']:<10} {corrected['name']:<11} {str(baseline['loaded']):>8} {str(corrected['loaded']):>9}  {corrected['reason']:<28} {corrected['checks']:>6} {str(corrected['correct']):>8}")  # 输出当前工具包的验证结论。
print("Provenance账本：")  # 标记下方为可审计来源链。
for record in provenance_ledger:  # 逐包展示来源、摘要、密钥和检查顺序。
    print(record)  # 输出当前工具包的脱敏 provenance 记录。
print(f"结果解读：只看签名字段准确率={baseline_accuracy:.1%}，真实验签加策略链={corrected_accuracy:.1%}；只有weather和docs被加载。")  # 解释密码学验证与运行策略缺一不可。

包         name        baseline  verified  reason                       checks  correct
pkg-01     weather         True      True  verified                          7     True
pkg-02     payments        True     False  permission_escalation             7     True
pkg-03     search          True     False  artifact_digest_mismatch          1     True
pkg-04     crm             True     False  revoked_key                       4     True
pkg-05     calendar        True     False  invalid_signature                 5     True
pkg-06     docs            True      True  verified                          7     True
pkg-07     analytics       True     False  schema_not_approved               6     True
Provenance账本：
{'package_id': 'pkg-01', 'name': 'weather', 'version': '1.4.2', 'publisher': 'acme-tools', 'key_id': 'acme-1', 'artifact_digest': 'af70bd78c9d74a26', 'decision': 'verified', 'check_chain': ['artifact_sha256', 'trusted_key', 'publisher_binding', 'key_revocation', 'rsa_signature', 'sc

## 5. 失败案例与修正：篡改权限后保留旧签名

对合法天气 manifest 增加 `filesystem.write`，旧 RSA 签名仍是非零整数；存在性基线继续接受，完整验证因签名覆盖的规范字节已改变而返回 `invalid_signature`。

In [5]:
tampered_manifest = {name: (value.copy() if isinstance(value, list) else value) for name, value in packages[0]["manifest"].items()}  # 浅拷贝合法天气 manifest 及权限列表。
tampered_manifest["permissions"] = sorted(tampered_manifest["permissions"] + ["filesystem.write"])  # 在签名后加入危险权限且不重新签名。
tampered_package = {"id": "pkg-tampered", "manifest": tampered_manifest, "delivered_artifact": packages[0]["delivered_artifact"], "expected": False}  # 构造制品摘要仍正确的权限篡改包。
shallow_accepts_tamper = bool(tampered_manifest.get("signature"))  # 复现只看 signature 字段的错误放行。
tampered_loaded, tampered_reason, tampered_trace = verify_package(tampered_package)  # 用受信公钥重新计算规范载荷验签。
tampered_signature_check = next(item for item in tampered_trace if item["check"] == "rsa_signature")  # 读取真实 RSA 恢复值和期望值。
print(f"错误行为：signature仍存在={shallow_accepts_tamper}，篡改权限={tampered_manifest['permissions']}，浅层检查会加载。")  # 展示签名存在性无法保证内容完整。
print(f"修正行为：loaded={tampered_loaded}，reason={tampered_reason}，RSA recovered={tampered_signature_check['recovered']}，expected={tampered_signature_check['expected']}")  # 展示实际模幂验签拒绝篡改载荷。

错误行为：signature仍存在=True，篡改权限=['filesystem.write', 'network.weather.read']，浅层检查会加载。
修正行为：loaded=False，reason=invalid_signature，RSA recovered=181，expected=3028


## 6. 生产边界

极小 RSA 可被瞬间破解，也没有 padding，绝不能上线。生产应使用 Ed25519/ECDSA 或 Sigstore keyless signing、规范格式版本、TUF root/targets 元数据、透明日志 inclusion proof、可信时间戳、KMS/HSM、阈值签名、撤销缓存与 sandbox 最小权限双重防线。

In [6]:
supply_chain_diagnostics = {"packages": len(packages), "loaded": sum(row["loaded"] for row in corrected_rows), "rejected": sum(not row["loaded"] for row in corrected_rows), "decision_accuracy": corrected_accuracy, "revoked_keys": len(revoked_key_ids), "provenance_records": len(provenance_ledger), "private_keys_in_runtime_verifier": 0, "production_grade_crypto": False}  # 汇总供应链决策、撤销和教学安全边界。
print("生产监控快照：", supply_chain_diagnostics)  # 输出工具加载器应持续监控的信号。

生产监控快照： {'packages': 7, 'loaded': 2, 'rejected': 5, 'decision_accuracy': 1.0, 'revoked_keys': 1, 'provenance_records': 7, 'private_keys_in_runtime_verifier': 0, 'production_grade_crypto': False}


## 7. 最小回归测试

断言覆盖样本规模、真实 RSA 验签、同数据决策、各类失败、权限篡改和 provenance 完整性。

In [7]:
assert len(packages) >= 5 and len(public_keyring) >= 2  # 保证案例包含足够工具包和多个发布者密钥。
assert first_verified and first_reason == "verified" and any(item["check"] == "rsa_signature" and item["passed"] for item in first_trace)  # 保证合法包经过真实 RSA 数学验签。
assert corrected_accuracy > baseline_accuracy and corrected_accuracy == 1.0  # 保证完整验证在同一批包上优于签名存在性。
assert {row["reason"] for row in corrected_rows if not row["loaded"]} == {"permission_escalation", "artifact_digest_mismatch", "revoked_key", "invalid_signature", "schema_not_approved"}  # 保证五种供应链失败都真实发生。
assert shallow_accepts_tamper and not tampered_loaded and tampered_reason == "invalid_signature"  # 保证保留旧签名不能绕过权限载荷篡改。
assert tampered_signature_check["recovered"] != tampered_signature_check["expected"]  # 保证拒绝来自真实 RSA 恢复值不匹配而非字段判断。
assert len(provenance_ledger) == len(packages) and all(record["artifact_digest"] and record["check_chain"] for record in provenance_ledger)  # 保证每个包都有摘要和检查链 provenance。